# Limitation DGVM value range to value range of training data

In [ ]:
# Libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'
dir04 = '../paper_ecosystem_carbon/output/04_dgvm_prep/'

---

In [ ]:
# List of ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

# List of used models
list_model = ['cablepop', 'classic', 'ibis', 'jsbach', 'jules', 'lpjguess', 
              'lpjml', 'ocn', 'orchidee', 'sdgvm', 'yibs']

In [ ]:
# Get translated dgvm data
ds_dgvm = xr.open_zarr(f'{dir04}ds_dgvm_translated_r025.zarr').compute()

# Get potential training data
ds_pot_train = xr.open_zarr(f'{dir02}ds_pot_train_r001.zarr')

# Get potential occurance data
ds_occur = xr.open_zarr(f'{dir02}ds_pot_occur_r025.zarr')

# Get carbon datasets
ds_cveg_biom = xr.open_zarr(f'{dir02}ds_cveg_biom_r001.zarr')
ds_cveg_tund = xr.open_zarr(f'{dir02}ds_cveg_tund_r001.zarr')
ds_cveg_crop = xr.open_zarr(f'{dir02}ds_cveg_crop_r001.zarr')
ds_cveg_gras = xr.open_zarr(f'{dir02}ds_cveg_gras_r001.zarr')
ds_csoil = xr.open_zarr(f'{dir02}ds_csoil_000_100_r001.zarr')

In [ ]:
def get_range_values(ctype, etype, p=True):

    """ Get the 1 and 99 percentiles of carbon values from potential training 
    data; Print percentiles if p is True"""

    # Define y_train (ensure consistency with 02_04_prep_database)
    if ctype == 'cveg':
        if etype in ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
                     'temp_dnf', 'glob_shrub']:
            da_c = ds_cveg_biom['cveg_biom']    
        if etype in ['glob_tundr', 'glob_mount']:
            da_c = ds_cveg_tund['cveg_tund']
        if etype in ['glob_grass', 'glob_past']:
            da_c = ds_cveg_gras['cveg_gras']
        if etype == 'glob_crop':
            da_c = ds_cveg_crop['cveg_crop']
    if ctype == 'csoil':
        da_c = ds_csoil['csoil_000_100']
    
    da_c_pot_train = da_c.where(ds_pot_train[etype])
    vmin = da_c_pot_train.min().compute().item()
    vmax = da_c_pot_train.max().compute().item()
    v001 = da_c_pot_train.quantile(0.01).compute().item()
    v099 = da_c_pot_train.quantile(0.99).compute().item()

    if p == True:
        pv001, pv099 = int(round(v001, 2)), int(round(v099, 2))
        pvmin, pvmax = int(round(vmin, 2)), int(round(vmax, 2))
        print(f'{ctype}_{etype} : {pv001}, {pv099} || {pvmin}, {pvmax}')
        
    return vmin, vmax, v001, v099

In [ ]:
# Exclude dgmv values that are not in the range of the 0.01 and 0.99 quantile 
# of the potential training dataset

# Create empty dataset
ds_dgvm_lim = xr.Dataset()

for ctype in ['cveg', 'csoil']:
    for etype in list_etype:

        # Get range values
        vmin, vmax, v001, v099 = get_range_values(ctype, etype)
        # Get data array
        da_dgvm = ds_dgvm[f'{ctype}_{etype}']
        # Exclude values above and below selected v values (v001, v099)
        # Limit to area of potential occurance
        da_dgvm_lim = da_dgvm \
            .where(da_dgvm > v001) \
            .where(da_dgvm < v099) \
            .where(ds_occur[etype])
        # Add variable to dataset
        ds_dgvm_lim[f'{ctype}_{etype}'] = da_dgvm_lim

In [ ]:
# Create chunks
ds_dgvm_lim = ds_dgvm_lim.chunk(model=1)

# Fix encoding, otherwise error when export
for var in ds_dgvm_lim.variables:
    ds_dgvm_lim[var].encoding.clear()
    
# Export dataset
ds_dgvm_lim.to_zarr(f'{dir04}ds_dgvm_limited_r025.zarr', mode='w');